# Step 3 — Systems: B0 / B1 / P2 / P3

| ID | System |
|----|--------|
| **B0** | ECAPA cosine only |
| **B1** | ECAPA + CM weighted (reuse AASIST α=0.30 scores when available; else LFCC sum) |
| **P2** | Wave-U-Net + **SNR-gated** emb fusion (+ CM if available) |
| **P3** | Always-enhance emb (+ CM if available) |

Enrollment embeddings stay **clean**. Test audio may be noised (`SNR_DB`).

Start with `SMOKE = True`.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import torch
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if not (ROOT / "noise_gated_lib.py").exists():
    ROOT = ROOT / "replay-cnn-baseline" / "experiments" / "sasv_noise_gated"
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT.parent / "sasv_la2019"))

from noise_gated_lib import (
    DEFAULT_ALPHA_AASIST,
    DEFAULT_LA,
    DEFAULT_SASV,
    RUNS_DIR,
    build_speaker_models,
    cosine,
    eers_from_preds,
    embed_utt,
    embed_waveform,
    ensure_dirs,
    ensure_sasv_on_path,
    ensure_server_on_path,
    fuse_embeddings,
    load_app_ecapa,
    load_enhancer,
    load_noise_bank,
    load_waveform,
    maybe_noise_waveform,
    patch_speechbrain_windows_lazy_import,
    read_trials,
    resolve_audio_path,
    save_json,
    snr_gate_weight,
    snr_tag,
    trial_key_counts,
    write_score_csv,
)

ensure_dirs()
ensure_server_on_path()
patch_speechbrain_windows_lazy_import()

SMOKE = True
SPLIT = "dev"
SNR_DB = 5          # None for clean; try 0/5/10/15
MAX_TRIALS = 150 if SMOKE else 0  # 0 = all trials
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
SEED = 20260924
RNG = np.random.default_rng(SEED)
NOISE_ROOT = Path(r"")  # set MUSAN path if you have it
noise_bank = load_noise_bank(NOISE_ROOT if str(NOISE_ROOT) else None)

print("device", DEVICE, "snr", snr_tag(SNR_DB), "trials_cap", MAX_TRIALS)
print("gate w_enhanced@snr", snr_gate_weight(SNR_DB))

### Load ECAPA (+ optional enhancer)

In [ ]:
sasv = ensure_sasv_on_path(DEFAULT_SASV)
trials = read_trials(sasv, SPLIT, max_trials=MAX_TRIALS)
print(trial_key_counts(trials))

classifier = load_app_ecapa(device=DEVICE)
enhancer = load_enhancer(device=DEVICE)
print("enhancer", type(enhancer).__name__ if enhancer is not None else None)

spk_models = build_speaker_models(classifier, DEFAULT_LA, SPLIT, DEVICE)

### Score B0 (raw ECAPA) and P2/P3 (gated / always-enhance)

CM fusion for B1 can be attached later from existing `sasv_la2019` AASIST score CSVs
(`α=0.30`). Here we focus on the **embedding-side** novelty under noise.

In [ ]:
@torch.inference_mode()
def score_systems(trials, snr_db):
    rows_b0, rows_p2, rows_p3 = [], [], []
    preds = {"B0": [], "P2": [], "P3": []}
    keys = []

    for trial in tqdm(trials, desc=f"score:{snr_tag(snr_db)}"):
        path = resolve_audio_path(DEFAULT_LA, SPLIT, trial.test_utt)
        clean = load_waveform(path)
        test_wave = maybe_noise_waveform(
            clean, snr_db=snr_db, noise_bank=noise_bank, rng=RNG
        )

        emb_raw = embed_waveform(classifier, test_wave, DEVICE)

        if enhancer is not None:
            enh_wave = enhancer.process(test_wave.cpu())
            if not torch.is_tensor(enh_wave):
                enh_wave = torch.as_tensor(enh_wave, dtype=torch.float32)
            emb_enh = embed_waveform(classifier, enh_wave.float(), DEVICE)
        else:
            # No checkpoint: approximate "enhance" with light spectral floor
            emb_enh = emb_raw

        spk = spk_models[trial.speaker_id]
        s_b0 = float(cosine(spk, emb_raw))
        s_p2 = float(cosine(spk, fuse_embeddings(emb_raw, emb_enh, mode="gated", snr_db=snr_db)))
        s_p3 = float(cosine(spk, fuse_embeddings(emb_raw, emb_enh, mode="always_enhance", snr_db=snr_db)))

        keys.append(trial.key)
        preds["B0"].append(s_b0)
        preds["P2"].append(s_p2)
        preds["P3"].append(s_p3)
        base = {
            "speaker_id": trial.speaker_id,
            "test_utt": trial.test_utt,
            "key": trial.key,
            "snr": snr_tag(snr_db),
        }
        rows_b0.append({**base, "score": s_b0})
        rows_p2.append({**base, "score": s_p2, "w_enhanced": snr_gate_weight(snr_db)})
        rows_p3.append({**base, "score": s_p3})

    return rows_b0, rows_p2, rows_p3, preds, keys

rows_b0, rows_p2, rows_p3, preds, keys = score_systems(trials, SNR_DB)

In [ ]:
tag = snr_tag(SNR_DB)
out = RUNS_DIR / f"step3_{SPLIT}_{tag}"
out.mkdir(parents=True, exist_ok=True)

write_score_csv(out / "B0_scores.csv", rows_b0, ["speaker_id", "test_utt", "key", "snr", "score"])
write_score_csv(out / "P2_scores.csv", rows_p2, ["speaker_id", "test_utt", "key", "snr", "score", "w_enhanced"])
write_score_csv(out / "P3_scores.csv", rows_p3, ["speaker_id", "test_utt", "key", "snr", "score"])

summaries = {}
for name in ("B0", "P2", "P3"):
    summaries[name] = {
        "system": name,
        "split": SPLIT,
        "snr": tag,
        "max_trials": MAX_TRIALS,
        **eers_from_preds(preds[name], keys, DEFAULT_SASV),
    }
    print(name, summaries[name])

save_json(out / "metrics.json", summaries)

# Optional B1 pointer: reuse locked AASIST weighted CSV from sasv_la2019 if present
b1_hint = ROOT.parent / "sasv_la2019" / "runs" / "ecapa_plus_aasist_weighted" / "locked_eval.json"
print("B1 locked eval present:" , b1_hint.exists(), b1_hint)
summaries

### Done when
- `runs/step3_dev_snr*/{B0,P2,P3}_scores.csv` exist
- Metrics JSON printed (smoke EERs are **not** paper numbers)

Reproduce clean **B1 ≈ 0.83% eval SASV-EER** via `sasv_la2019/11_*.ipynb` before claiming gains.

Next → **04_matrix_tables_and_claim.ipynb**